# Tokenization

Before a transformer can process text, every character needs to become a number.
That is all a tokenizer does — it builds a lookup table from the corpus and uses
it to convert back and forth between text and integers.

We use a **character-level** tokenizer: each unique character in Shakespeare gets
its own ID. The full vocab is 65 characters — lowercase, uppercase, punctuation,
space, and newline.

In [12]:
import plotly.io as pio

from protoform.data.shakespeare import load_encoded, load_split
from protoform.tokenizer import CharTokenizer

pio.renderers.default = "notebook"  # ensures text/html output for Sphinx

# Build vocab from the training split
full_text = load_split("train")
tok = CharTokenizer(full_text)

# Encode once — all training works on these tensors
splits = load_encoded(tok)

print(f"Vocab size   : {tok.vocab_size} unique characters")
print(f"Vocab        : {''.join(sorted(set(full_text)))!r}")
print()
print(f"Train tokens : {len(splits['train']):,}")
print(f"Val tokens   : {len(splits['validation']):,}")
print(f"Test tokens  : {len(splits['test']):,}")

## Encoding — text to integers

Each character maps to a fixed integer ID. The same character always gets the
same ID, no matter where it appears in the text.

In [ ]:
passage = "First Citizen:"
ids = tok.encode(passage)

print(f"Text : {passage}")
print(f"IDs  : {ids}")
print()
for char, token_id in zip(passage, ids, strict=True):
    print(f"  '{char}' -> {token_id}")

## Decoding — integers back to text

The reverse mapping is equally simple. This is how a trained model turns its
predicted token IDs back into readable text.

In [ ]:
recovered = tok.decode(ids)
print(f"IDs      : {ids}")
print(f"Decoded  : {recovered}")
print(f"Roundtrip: {recovered == passage}")

## Real Shakespeare

Let's encode the opening lines of the dataset and see what the model actually
receives as input.

In [ ]:
# Show the opening of the training data as text and as token IDs
opening_ids = splits["train"][:100]
opening_text = tok.decode(opening_ids.tolist())

print("Original text:")
print(opening_text)
print()
print("As token IDs:")
print(opening_ids.tolist())

## Visualising the vocabulary

Each cell is one character. Its colour shows how frequently it appears in the
training text — darker means more common.

In [ ]:
import plotly.graph_objects as go

vocab = sorted(set(full_text))
counts = [full_text.count(c) for c in vocab]
labels = [repr(c) for c in vocab]

fig = go.Figure(
    go.Bar(
        x=labels,
        y=counts,
        marker_color=counts,
        marker_colorscale="Blues",
    )
)
fig.update_layout(
    title="Character frequency in the Shakespeare training corpus",
    xaxis_title="character",
    yaxis_title="count",
    height=400,
)
fig.show()